In [2]:
from catboost import CatBoostClassifier

loaded_model = CatBoostClassifier()

loaded_model.load_model(
    "saved_models/catboost_final.cbm"
)

print("Model loaded successfully!")

Model loaded successfully!


In [4]:
print("Model Type:", type(loaded_model).__name__)

print("Tree Count:", loaded_model.tree_count_)

print("Number of Features:",
      len(loaded_model.feature_names_))

print("Feature Names:")
print(loaded_model.feature_names_)

print("\nModel Parameters:")

params = loaded_model.get_all_params()

for parameter, value in params.items():
    print(f"{parameter}: {value}")

Model Type: CatBoostClassifier
Tree Count: 1000
Number of Features: 73
Feature Names:
['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72']

Model Parameters:
nan_mode: Min
gpu_ram_part: 0.95
eval_metric: TotalF1
iterations: 1000
leaf_estimation_method: Newton
observations_to_bootstrap: TestOnly
random_score_type: NormalWithModelSizeDecrease
grow_policy: SymmetricTree
penalties_coefficient: 1
boosting_type: Plain
feature_border_type: GreedyLogSum
bayesian_matrix_reg: 0.1000000015
devices: 0
eval_fraction: 0
pinned_memory_bytes: 104857600
force_unit_auto_pair_weights: False
l2_leaf_reg: 3
random_strength: 1
rsm: 1
boos

In [7]:
print("Number of features:",
      len(loaded_model.feature_names_))

print("Feature names:",
      loaded_model.feature_names_)

Number of features: 73
Feature names: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72']


In [2]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
import joblib

# Load saved model
loaded_model = CatBoostClassifier()
loaded_model.load_model(
    "saved_models/catboost_final.cbm"
)

# Load preprocessor assets and label encoder
assets = joblib.load("saved_models/preprocessing_assets.pkl")
preprocessor = assets['preprocessor']
country_freq = assets['country_freq']
common_species = assets['common_species']
label_encoder = joblib.load("saved_models/label_encoder.pkl")

# Enter custom input using the original 12 feature columns
# disease_name is the target, so it is not included
custom_input = pd.DataFrame([{
    "species_name": "Swine",
    "is_wild": 0,
    "is_domestic": 1,
    "is_aquatic": 0,
    "causal_agent_type": "Virus",
    "country_name": "Bolivia",
    "latitude": -19.7444,
    "longitude": -64.1010,
    "month": "August",
    "season": "Monsoon",
    "susceptible": 1461,
    "epi_unit_type": "Farm"
}])

# Manual Preprocessing Steps
# 1. Clean species name
custom_input["species_name"] = (
    custom_input["species_name"]
    .str.replace(r"\s*\([^)]*\)", "", regex=True)
    .str.replace("\n", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# 2. Common species
custom_input["species_name"] = custom_input["species_name"].where(
    custom_input["species_name"].isin(common_species),
    "Other"
)

# 3. Country frequency
custom_input["country_freq"] = custom_input["country_name"].map(country_freq).fillna(0)
custom_input = custom_input.drop(columns=["country_name"])

# 4. Month encoding
month_map = {
    "January": 1, "February": 2, "March": 3, "April": 4, "May": 5, "June": 6,
    "July": 7, "August": 8, "September": 9, "October": 10, "November": 11, "December": 12
}
custom_input["month_num"] = custom_input["month"].map(month_map)
custom_input["month_sin"] = np.sin(2 * np.pi * custom_input["month_num"] / 12)
custom_input["month_cos"] = np.cos(2 * np.pi * custom_input["month_num"] / 12)
custom_input = custom_input.drop(columns=["month", "month_num"])

# 5. Susceptible log transform
custom_input["susceptible"] = np.log1p(custom_input["susceptible"])

# Apply the ColumnTransformer preprocessing
custom_input_encoded = preprocessor.transform(custom_input)

# Predict
prediction = loaded_model.predict(custom_input_encoded)

# Convert encoded prediction into disease name
predicted_disease = label_encoder.inverse_transform(
    [int(prediction[0][0])]
)[0]

print("Predicted Disease:", predicted_disease)

Predicted Disease: Aujeszky's disease
